# Screening: Descriptive Statistics

Descriptive statistics for calculated and converted columns from `SubjectScreening`,
filtered to eligible subjects only (`eligible == True`).

In [ ]:
%%capture
import os

import numpy as np
import pandas as pd
from dj_notebook import activate
from tabulate import tabulate

env_file = os.environ["META_ENV"]
plus = activate(dotenv_file=env_file)
pd.set_option("future.no_silent_downcasting", True)

In [ ]:
from django_pandas.io import read_frame
from meta_prn.models import EndOfStudy

from meta_screening.models import SubjectScreening

cols = [
    "screening_identifier",
    "subject_identifier",
    "eligible",
    "calculated_egfr_value",
    "converted_creatinine_value",
    "converted_fbg_value",
    "converted_fbg2_value",
    "converted_ogtt_value",
    "converted_ogtt2_value",
    "repeat_glucose_performed",
    "revision",
    "eligibility_datetime",
]

qs = SubjectScreening.objects.filter(eligible=True).values(*cols)
df = read_frame(qs, verbose=False)
print(f"Eligible subjects: {len(df)}")

In [ ]:
df["eligibility_datetime"] = pd.to_datetime(df["eligibility_datetime"], utc=True)

In [ ]:
numeric_cols_orig = [
    "calculated_egfr_value",
    "converted_creatinine_value",
    "converted_fbg_value",
    "converted_fbg2_value",
    "converted_ogtt_value",
    "converted_ogtt2_value",
]

df[numeric_cols_orig] = df[numeric_cols_orig].apply(pd.to_numeric, errors="coerce").astype("Float64")

repeat_mask = (df["repeat_glucose_performed"] == "Yes")

df["fbg"] = df["converted_fbg_value"]
df["ogtt"] = df["converted_ogtt_value"]

df.loc[df["converted_fbg_value"].isna() & repeat_mask & df["converted_fbg2_value"].notna(), "fbg"] = df["converted_fbg2_value"]
df.loc[df["converted_ogtt_value"].isna() & repeat_mask & df["converted_ogtt2_value"].notna(), "ogtt"] = df["converted_ogtt2_value"]

# df = df.drop(columns=["converted_fbg2_value", "converted_ogtt2_value", "repeat_glucose_performed"])

numeric_cols = [
    "calculated_egfr_value",
    "converted_creatinine_value",
    "fbg",
    "ogtt",
]

In [ ]:
df_stats = df[numeric_cols].describe().T

df_stats["missing"] = len(df) - df[numeric_cols].notna().sum()
df_stats["missing_%"] = (df_stats["missing"] / len(df) * 100).round(1)

df_stats = df_stats[["count", "missing", "missing_%", "mean", "std", "min", "25%", "50%", "75%", "max"]]
df_stats = df_stats.round(2)

print(
    tabulate(
        df_stats,
        headers=df_stats.columns,
        tablefmt="grid",
        floatfmt=".2f",
    )
)

In [ ]:
# # were the repeat values used?
# # these subjects are not eligible based on the repeat
# scr_ids = df[
#     ((df["converted_fbg_value"]<6.1) | (df["converted_fbg_value"]>6.9)) &
#     ((df["converted_ogtt_value"]<7.0) | (df["converted_ogtt_value"]>11.1))
# ][["screening_identifier", "subject_identifier"]]

In [ ]:
# check eligibility by glucose is correct
# the 15 that appear here are eligible by either a FBG-PRE + OGTT-NORMAL or FBG-NORMAL + OGTT-PRE
# see EligibilityPartThreePhaseThree docstring for complete definition

# mask1 = (
#         (
#             ((df["converted_fbg_value"]>=6.1) & (df["converted_fbg_value"]<=6.9)) |
#             ((df["converted_fbg2_value"]>=6.1) & (df["converted_fbg2_value"]<=6.9))
#          ) &
#         (
#             ((df["converted_ogtt_value"]>=7.0) & (df["converted_ogtt_value"]<=11.1)) |
#             ((df["converted_ogtt2_value"]>=7.0) & (df["converted_ogtt2_value"]<=11.1))
#         )
#     )
# df[(df.screening_identifier.isin(scr_ids.screening_identifier)) & ~mask1][["screening_identifier", "subject_identifier", *numeric_cols_orig]]

In [ ]:
# df[(df.screening_identifier.isin(scr_ids.screening_identifier)) & ~mask1][["screening_identifier", "subject_identifier", "eligible",     "revision",
#     "eligibility_datetime",
# *numeric_cols_orig]]

In [ ]:
# scr_ids

In [ ]:
# df_eos = read_frame(EndOfStudy.objects.all(), verbose=True)

In [ ]:
# df_eos[df_eos.subject_identifier.isin(scr_ids.subject_identifier)]

In [ ]:
# # rerun all through EligibilityPartThreePhaseThree
# from meta_screening.eligibility import EligibilityPartThreePhaseThree
# from clinicedc_constants import YES
#
# p3 = {}
#
# for model_obj in SubjectScreening.objects.filter(screening_identifier__in=list(scr_ids.screening_identifier)):
#     p = EligibilityPartThreePhaseThree(
#             model_obj=model_obj,
#             update_model=False,
#         )
#     if p.eligible == YES:
#         p3.update({model_obj.screening_identifier: p})
#
